# Chapter 5 Lab: Cross‑Validation and the Bootstrap

This notebook reproduces the resampling examples from Section 5.3 of *An Introduction to Statistical Learning* (Python version).  
We cover:

- The validation set approach
- *K*‑fold cross‑validation and LOOCV
- The bootstrap for standard errors

All code is adapted from the `ISLP` lab and uses `statsmodels`, `sklearn`, and `numpy`.

## 1. Imports

We start with the necessary libraries. Some are new compared to earlier chapters.

In [1]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize, poly)
from sklearn.model_selection import train_test_split
from functools import partial
from sklearn.model_selection import cross_validate, KFold, ShuffleSplit
from sklearn.base import clone
from ISLP.models import sklearn_sm

## 2. The Validation Set Approach

We use the Auto data set to predict mpg using horsepower.<br>
We split the data into a training set (196 observations) and a validation set (196 observations).<br>
We then evaluate linear, quadratic, and cubic polynomial fits.

### 2.1 Load the data and split

In [2]:
Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto, test_size=196, random_state=0)

### 2.2 Fit a linear model on the training set

In [3]:
hp_ms = MS(['horsepower'])
X_train = hp_ms.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

X_valid = hp_ms.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
mse_linear = np.mean((y_valid - valid_pred)**2)
print(f"Validation MSE (linear): {mse_linear:.4f}")

Validation MSE (linear): 23.6166


### 2.3 Helper function to compute validation MSE for polynomial fits

In [4]:
def evalMSE(terms, response, train, test):
    mm = MS(terms)
    X_train = mm.fit_transform(train)
    y_train = train[response]
    X_test = mm.transform(test)
    y_test = test[response]
    results = sm.OLS(y_train, X_train).fit()
    test_pred = results.predict(X_test)
    return np.mean((y_test - test_pred)**2)

### 2.4 Compare degrees 1, 2, and 3 (first split)

In [5]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)], 'mpg', Auto_train, Auto_valid)
print("Validation MSEs for degrees 1,2,3 (split 1):", MSE)

Validation MSEs for degrees 1,2,3 (split 1): [23.61661707 18.76303135 18.79694163]


### 2.5 Repeat with a different random split

In [6]:
Auto_train2, Auto_valid2 = train_test_split(Auto, test_size=196, random_state=3)
MSE2 = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE2[idx] = evalMSE([poly('horsepower', degree)], 'mpg', Auto_train2, Auto_valid2)
print("Validation MSEs for degrees 1,2,3 (split 2):", MSE2)

Validation MSEs for degrees 1,2,3 (split 2): [20.75540796 16.94510676 16.97437833]


*Observation: The quadratic model gives the lowest validation error; adding cubic terms does not improve.*

## 3. Cross‑Validation

We use sklearn’s cross_validate together with our wrapper sklearn_sm to perform LOOCV and 10‑fold CV.

### 3.1 LOOCV (Leave‑One‑Out Cross‑Validation)

In [7]:
hp_model = sklearn_sm(sm.OLS, MS(['horsepower']))
X = Auto.drop(['mpg'], axis=1)
Y = Auto['mpg']

cv_results = cross_validate(hp_model, X, Y, cv=Auto.shape[0])
loocv_err = np.mean(cv_results['test_score'])
print(f"LOOCV MSE (linear): {loocv_err:.4f}")

LOOCV MSE (linear): 24.2315


### 3.2 LOOCV for polynomial degrees 1 to 5

In [8]:
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
cv_error = np.zeros(5)

for i, d in enumerate(range(1, 6)):
    X_poly = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M, X_poly, Y, cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])

print("LOOCV MSE for degrees 1‑5:", cv_error)

LOOCV MSE for degrees 1‑5: [24.23151352 19.24821312 19.33498406 19.4244303  19.03320357]


### 3.3 10‑Fold Cross‑Validation

In [9]:
cv = KFold(n_splits=10, shuffle=True, random_state=0)
cv_error_10 = np.zeros(5)

for i, d in enumerate(range(1, 6)):
    X_poly = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M, X_poly, Y, cv=cv)
    cv_error_10[i] = np.mean(M_CV['test_score'])

print("10‑fold CV MSE for degrees 1‑5:", cv_error_10)

10‑fold CV MSE for degrees 1‑5: [24.20766449 19.18533142 19.27626666 19.47848401 19.13719864]


### 3.4 Using ShuffleSplit for the validation set approach

In [10]:
validation = ShuffleSplit(n_splits=1, test_size=196, random_state=0)
results = cross_validate(hp_model, X, Y, cv=validation)
print("Validation MSE (single split) via ShuffleSplit:", results['test_score'][0])

Validation MSE (single split) via ShuffleSplit: 23.61661706966988


### 3.5 Estimate variability over multiple validation splits

In [11]:
validation10 = ShuffleSplit(n_splits=10, test_size=196, random_state=0)
results10 = cross_validate(hp_model, X, Y, cv=validation10)
print(f"Mean validation MSE (10 splits): {results10['test_score'].mean():.4f}")
print(f"Std deviation: {results10['test_score'].std():.4f}")

Mean validation MSE (10 splits): 23.8022
Std deviation: 1.4218


## 4. The Bootstrap

We illustrate the bootstrap on two examples:

1. Estimating the standard error of the optimal portfolio allocation α.
2. Estimating the standard errors of regression coefficients in a linear model.

### 4.1 Bootstrap for the Portfolio α

The Portfolio data set contains returns of two assets.<br>
We define a function that computes the minimum‑variance weight α:

In [12]:
Portfolio = load_data('Portfolio')

def alpha_func(D, idx):
    cov = np.cov(D[['X','Y']].iloc[idx], rowvar=False)
    return ((cov[1,1] - cov[0,1]) / (cov[0,0] + cov[1,1] - 2*cov[0,1]))

#### 4.1.1 One bootstrap sample

In [13]:
rng = np.random.default_rng(0)
alpha_bs = alpha_func(Portfolio, rng.choice(100, 100, replace=True))
print("α from one bootstrap sample:", alpha_bs)

α from one bootstrap sample: 0.6074452469619004


#### 4.1.2 Bootstrap standard error function

In [14]:
def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        #idx = rng.choice(D.index, n, replace=True)
        idx = rng.choice(D.shape[0], n, replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

#### 4.1.3 Compute SE of α with B=1000

In [15]:
alpha_SE = boot_SE(alpha_func, Portfolio, B=1000, seed=0)
print(f"Bootstrap SE of α: {alpha_SE:.4f}")

Bootstrap SE of α: 0.0912


### 4.2 Bootstrap for linear regression coefficients
We want to estimate the standard errors of the intercept and slope when regressing mpg on horsepower.

#### 4.2.1 Helper function to bootstrap a regression

In [16]:
def boot_OLS(model_matrix, response, D, idx):
    D_ = D.iloc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_, X_).fit().params

#### 4.2.2 Freeze the model specification using partial

In [17]:
hp_func = partial(boot_OLS, MS(['horsepower']), 'mpg')

#### 4.2.3 Quick check: 10 bootstrap samples

In [18]:
rng = np.random.default_rng(0)
samples = np.array([hp_func(Auto, rng.choice(392, 392, replace=True)) for _ in range(10)])
print("First 10 bootstrap coefficient estimates:\n", samples)

First 10 bootstrap coefficient estimates:
 [[39.88064456 -0.1567849 ]
 [38.73298691 -0.14699495]
 [38.31734657 -0.14442683]
 [39.91446826 -0.15782234]
 [39.43349349 -0.15072702]
 [40.36629857 -0.15912217]
 [39.62334517 -0.15449117]
 [39.0580588  -0.14952908]
 [38.66688437 -0.14521037]
 [39.64280792 -0.15555698]]


#### 4.2.4 Bootstrap standard errors for the linear model

In [19]:
hp_se = boot_SE(hp_func, Auto, B=1000, seed=10)
print("Bootstrap SEs (intercept, horsepower):")
print(hp_se)

Bootstrap SEs (intercept, horsepower):
intercept     0.848807
horsepower    0.007352
dtype: float64


#### 4.2.5 Compare with the usual statsmodels standard errors

In [20]:
hp_model = sm.OLS(Auto['mpg'], MS(['horsepower']).fit_transform(Auto))
results = hp_model.fit()
summ = summarize(results)
print("Standard errors from statsmodels:")
print(summ['std err'])

Standard errors from statsmodels:
intercept     0.717
horsepower    0.006
Name: std err, dtype: float64


*Note: The bootstrap standard errors are slightly larger because the linear model is misspecified (non‑linear relationship).<br>
The bootstrap does not rely on the same assumptions as the theoretical formulas.*

### 4.3 Bootstrap for the quadratic model

In [21]:
quad_model = MS([poly('horsepower', 2, raw=True)])
quad_func = partial(boot_OLS, quad_model, 'mpg')
quad_se = boot_SE(quad_func, Auto, B=1000, seed=0)
print("Bootstrap SEs for quadratic model:")
print(quad_se)

# Standard errors from statsmodels
X_quad = quad_model.fit_transform(Auto)
quad_fit = sm.OLS(Auto['mpg'], X_quad).fit()
print("Standard errors from statsmodels (quadratic):")
print(summarize(quad_fit)['std err'])

Bootstrap SEs for quadratic model:
intercept                                  2.067840
poly(horsepower, degree=2, raw=True)[0]    0.033019
poly(horsepower, degree=2, raw=True)[1]    0.000120
dtype: float64
Standard errors from statsmodels (quadratic):
intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64


## 5. Summary

The validation set approach is simple but highly variable.

Cross‑validation (LOOCV and K‑fold) provides more stable test error estimates.

The bootstrap is a powerful tool for estimating the variability of any statistic, and it often gives more accurate standard errors when model assumptions are violated.

All code above follows the examples from the ISLP lab.
Remember that your results may differ slightly due to random seeds; the outputs shown in the book are reproduced with the seeds given.